# Candidate Ranking & Reasoning Dry Run

This notebook demonstrates loading, filtering, ranking, and generating factual reasonings for candidates.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import stream_candidates
from src.filters import should_filter_out
from src.scorer import CandidateScorer
from src.reasoner import generate_reasoning

## Run Ranker Pipeline on Small Slice

We'll simulate running the ranking logic on the first 500 candidates and outputting the top 5 ranked candidates with reasonings.

In [ ]:
CANDIDATES_PATH = '../data/raw/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl'
scorer = CandidateScorer()
jd_text = "Senior AI Engineer with embeddings, retrieval, and vector search experience at product startups."

results = []
count = 0

for c in stream_candidates(CANDIDATES_PATH):
    if should_filter_out(c):
        continue
    score = scorer.score_candidate(c, jd_text)
    results.append({
        "candidate_id": c["candidate_id"],
        "score": score,
        "candidate": c
    })
    
    count += 1
    if count >= 1000:  # Scan first 1000 candidates
        break

# Sort by score descending and candidate_id ascending
results.sort(key=lambda x: (-x["score"], x["candidate_id"]))

# Show top 5
print("=== Top 5 Ranked Candidates ===")
for i, item in enumerate(results[:5]):
    rank = i + 1
    reason = generate_reasoning(item["candidate"], rank, item["score"])
    print(f"Rank {rank}: {item['candidate_id']} | Score: {item['score']:.4f}")
    print(f"Reasoning: {reason}")
    print("-" * 40)